# M5a — Portfolio Ising reference (PennyLane Lightning CPU)

**Research question:** Can a small synthetic portfolio QUBO be mapped to a Pauli-Z Ising Hamiltonian and evaluated with a local VQE-style statevector ansatz?

This notebook uses **upstream PennyLane/Lightning and NumPy/SciPy**, not a new workflow SDK. It compares an analytic (shots=None) quantum-state expectation to an independently enumerated classical optimum. The optimizer is not assumed to find the global optimum.

It does **not** request H200, a remote QPU, Slurm, provider credentials, or use GPU inference capacity. Install in a Python 3.14 environment with `pip install -e '.[lightning-cpu]'` from the repository root. On shared systems, use a user-approved environment.

In [ ]:
from pathlib import Path
from quantum_workflows.portfolio_ising import (
    reference_problem, exact_reference, ising_energy, run_lightning_cpu
)

problem = reference_problem()  # synthetic expected returns + covariance, budget=1
reference = exact_reference(problem)
print('Exact optimum holdings:', reference['optimal_bits'])
print('Classical minimum energy:', reference['minimum_energy'])
print('Pauli-Z energy for reference:', ising_energy(problem, tuple(reference['optimal_bits'])))

In [ ]:
summary, result_dir = run_lightning_cpu(
    problem=problem, output=Path('./results'), seed=7, restarts=4, maxiter=180
)
print('Local device:', summary['backend'])
print('Estimated analytic energy:', summary['optimized_energy'])
print('Exact enumerated minimum:', summary['exact_reference_energy'])
print('Optimizer gap (reported, not hidden):', summary['optimality_gap'])
print('Artifact directory:', result_dir)

In [ ]:
import json
manifest = json.loads((result_dir / 'manifest.json').read_text())
print('Recorded executor:', manifest['runtime']['executor'])
print('Provenance commit:', manifest['source']['commit'])
print('Artifact hashes:', [(a['path'], a['sha256']) for a in manifest['artifacts']])

## Interpretation and next gate

The portfolio objective uses `-μᵀx + ρxᵀΣx + A(Σx − B)²` with binary holdings. `x=(1−Z)/2` gives the diagonal Ising Hamiltonian. The reference exhaustively enumerates `2^n` candidates. An analytic expectation can be above the true minimum; **that is not a numerical error or a claimed quantum speedup**.

The next gate, M5b, requires a **real, independently approved H200 allocation**, pinned CUDA/cuQuantum/PennyLane Lightning GPU runtime, actual scheduler job ID, and CPU↔GPU numerical agreement. No device access is implied by this CPU notebook.